# Finding substances unique to the AIIC
This notebook demonstrates how to query a compiled global inventory to identify substances that appear on the AIIC but not on comparable inventories from other jurisdictions. Such queries are useful for identifying jurisdiction-specific substances.

**Prerequisites**
- Download a copy of the [example global inventory](https://github.com/envris/global-inventory-compiler/tree/main/data) from the GIC repository.

## Step 1: Load the inventory and inspect the available list tags
Create an Inventory instance and print the list tags present in the dataset.

In [ ]:
from gic import Inventory

# Load the precompiled inventory.
inv = Inventory("input/global_inventory.jsonl")
all_list_tags = list(inv.get_list_data().values())

print(all_list_tags)

['JP-ISHA', 'PH-PICCS', 'US-TSCA_5F', 'CA-DSL', 'CN-IECSC', 'AU-ICHEMS_S6', 'KR-KECL_T', 'JP-CSCL', 'US-TSCA_TP', 'EU-REACH_PR', 'US-TSCA_CDR', 'CL-HSIUN', 'EU-ARN', 'JP-PRTR_SDS', 'US-TSCA_R', 'AU-NPI', 'TH-HAS', 'KR-KECL_R', 'TR-CSI', 'EU-POPs', 'MY-EHS', 'JP-CSCL_CI', 'KR-KECL_PP', 'TR-CSI_SVHC', 'JP-CSCL_CII', 'EU-CoRAP', 'NZ-NZIoC', 'EU-REACH_SVHC', 'EU-REACH_R', 'TW-CSI', 'US-TSCA_12C', 'EU-SPIN', 'JP-CSCL_PAC', 'KR-KECL', 'EU-REACH_XIV', 'AU-AIIC', 'US-TSCA_T', 'AU-ICHEMS_S7', 'AU-ICHEMS_S5', 'JP-CSCL_M', 'EU-REACH_XVII', 'US-TSCA', 'KR-KECL_SSIC', 'KR-KECL_PR', 'EU-CLP_VI', 'US-TSCA_5E', 'US-TRI', 'CASCC', 'PubChem']


## Step 2: Define the relevant and irrelevant list tags
By default, this query would treat any additional list association as evidence that a substance is not unique to the AIIC. For this use case, however, many list tags represent specific regulatory controls rather than national inventories, so they should be excluded from the uniqueness test.

In [ ]:
# Major inventory list tags to include in the comparison.
relevant_tags = [
    "AU-AIIC",  # Australia
    "CA-DSL",  # Canada
    "CL-HSIUN",  # Chile
    "CN-IECSC",  # China
    "EU-REACH_PR",  # Europe
    "EU-REACH_R",  # Europe
    "JP-CSCL",  # Japan
    "KR-KECL",  # South Korea
    "NZ-NZIoC",  # New Zealand
    "TW-CSI",  # Taiwan
    "US-TSCA",  # United States
]

# Tags outside this set are ignored when assessing uniqueness.
irrelevant_tags = [tag for tag in all_list_tags if tag not in relevant_tags]
print(irrelevant_tags)

['JP-ISHA', 'PH-PICCS', 'US-TSCA_5F', 'AU-ICHEMS_S6', 'KR-KECL_T', 'US-TSCA_TP', 'US-TSCA_CDR', 'EU-ARN', 'JP-PRTR_SDS', 'US-TSCA_R', 'AU-NPI', 'TH-HAS', 'KR-KECL_R', 'TR-CSI', 'EU-POPs', 'MY-EHS', 'JP-CSCL_CI', 'KR-KECL_PP', 'TR-CSI_SVHC', 'JP-CSCL_CII', 'EU-CoRAP', 'EU-REACH_SVHC', 'US-TSCA_12C', 'EU-SPIN', 'JP-CSCL_PAC', 'EU-REACH_XIV', 'US-TSCA_T', 'AU-ICHEMS_S7', 'AU-ICHEMS_S5', 'JP-CSCL_M', 'EU-REACH_XVII', 'KR-KECL_SSIC', 'KR-KECL_PR', 'EU-CLP_VI', 'US-TSCA_5E', 'US-TRI', 'CASCC', 'PubChem']


## Step 3: Run the query
The *get_by_list_tags_unique* method returns identifiers for substances associated only with the selected list tags. In this case, the *to_ignore* argument is used to exclude list tags that are not relevant to cross-inventory uniqueness.

In [ ]:
aiic_sids = inv.get_by_list_tags("sid", ["AU-AIIC"])  # Basic list query for reference
unique_sids = inv.get_by_list_tags_unique("sid", ["AU-AIIC"], to_ignore=irrelevant_tags)

print(f"Number of substances in inventory: {len(inv.get_sids())}")
print(f"Number of substances on the AIIC: {len(aiic_sids)}")
print(f"Number of substances unique to the AIIC: {len(unique_sids)}")

Number of substances in inventory: 320220
Number of substances on the AIIC: 39433
Number of substances unique to the AIIC: 4985


## Step 4: Export the results (optional)
SIDs are assigned during compilation and have limited meaning outside the context of a specific inventory. Exporting the matching records makes them easier to inspect and share.

In [ ]:
results = inv.export_as_data_frame(unique_sids)

# Save the results for inspection.
results.to_csv("output/unique_substances.csv")
results.to_xlsx("output/unique_substances.xlsx")